In [1]:
import sys
sys.path.append('../')

%env MUJOCO_GL=egl

env: MUJOCO_GL=egl


In [2]:
import jax
import jax.numpy as jnp
import mujoco
import numpy as np
import mediapy as media
from dataclasses import dataclass, field
from mujoco.mjx._src import math as mjx_math

from builderbench.env_utils import make_env
from utils.wrapper import wrap_env

In [3]:
@dataclass
class Args:
    # experiment
    agent: str = "mp"
    seed: int = 1

    # environment
    env_id: str = 'sparse-creative-4-task1'
    env_early_termination: bool = True
    env_episode_length: int = None
    permutation_invariant_reward: bool = True   # invariance to the order of cubes in any structure

    # planner
    kp_pos: float = 10.0
    kd_pos: float = 2.0

    kp_yaw: float = 10.0
    kd_yaw: float = 0.5

    duration: int = 20
    
    num_envs: str = 1

In [4]:
args = Args()

In [5]:
from utils.wrapper import Wrapper

@jax.jit
def get_yaw_from_quat(q):
    w, x, y, z = q[0], q[1], q[2], q[3]
    siny_cosp = 2 * (w * z + x * y)
    cosy_cosp = 1 - 2 * (y * y + z * z)
    yaw = jnp.arctan2(siny_cosp, cosy_cosp)
    return yaw
    
class PDWrapper(Wrapper):
    def __init__(self, env, duration: int = 1, kp_pos: float = 10.0, kd_pos: float = 2.0, kp_yaw: float = 10.0, kd_yaw: float = 0.5, delta_control: bool = False):
        super().__init__(env)
        
        self._kp_pos = kp_pos
        self._kd_pos = kd_pos
        self._kp_yaw = kp_yaw
        self._kd_yaw = kd_yaw
        self._duration = duration
        self._delta_control = delta_control
        
        self._cube_mass = 0.07936
        self._gravity = 9.81
        self._gravity_comp = self._cube_mass * self._gravity

        self._workspace_median = (self.env._workspace_bounds[1] + self.env._workspace_bounds[0]) / 2
        self._workspace_halfspan = (self.env._workspace_bounds[1] - self.env._workspace_bounds[0]) / 2

        self._yaw_median = jnp.array( [0.0] )
        self._yaw_halfspan = jnp.array( [np.pi / 2] )

    def get_action(self, state, waypoint_pos, waypoint_yaw, cube_id):
        current_pos = state.data.qpos[self.env._objs_qposadr[:, None] + np.arange(3)][cube_id]
        current_quat = state.data.qpos[(self.env._objs_qposadr + 3)[:, None] + np.arange(4)][cube_id]
        current_linvel = state.data.qvel[self.env._objs_qveladr[:, None] + np.arange(3)][cube_id]
        current_angvel = state.data.qvel[(self.env._objs_qveladr + 3)[:, None] + np.arange(3)][cube_id]
        
        error_pos = waypoint_pos - current_pos
        output_pos = (self._kp_pos * error_pos) + (self._kd_pos * - current_linvel)
        output_pos = output_pos.at[2].add(self._gravity_comp)

        current_yaw = get_yaw_from_quat(current_quat)
        delta_yaw = waypoint_yaw - current_yaw
        error_yaw = jnp.arctan2(jnp.sin(delta_yaw), jnp.cos(delta_yaw))        
        output_yaw = (self._kp_yaw * error_yaw) + (self._kd_yaw * - current_angvel[-1])

        raw_ctrl_action = jnp.concatenate([output_pos, output_yaw], axis=0)
        ctrl_action = ( raw_ctrl_action - self.env._ctrl_median[:4] ) / self.env._ctrl_halfspan[:4]
        
        select_action = ( ( ( 2 * cube_id + 1) * jnp.pi / self.env._config.num_cubes ) - jnp.pi ) / ( jnp.pi )

        action =  jnp.concatenate([ctrl_action, select_action[None]], axis=0)
        action = jnp.clip(action, -1, 1)
        return action
    
    def step(self, state, action):

        state.info.update(
            select_action = jnp.clip(action[-1], -1, 1),
        )
        cube_id = jnp.digitize( ( self.env._action_scale[-1] * action[-1] + jnp.pi ), bins = jnp.arange(1, self.env._config.num_cubes+1) * 2 * jnp.pi / ( self.env._config.num_cubes ) )

        if self._delta_control:
            raise NotImplementedError
        else:
            waypoint_pos = action[:3] * self._workspace_halfspan + self._workspace_median
            waypoint_yaw = action[3] * self._yaw_halfspan + self._yaw_median

        def f(carry, _):
            state, prev_done = carry
            action = self.get_action(state, waypoint_pos, waypoint_yaw, cube_id)
            state = self.env.step(state, action)
            done = jnp.maximum(state.done, prev_done)
            
            return (state, done), (state.reward, prev_done, state.metrics)
        
        (state, _), (rewards, dones, metrics) = jax.lax.scan(f, (state, state.done), (), self._duration)

        state = state.replace(reward = jnp.sum(rewards * (1-dones)))
        state = state.replace(metrics = jax.tree_util.tree_map(lambda m: jnp.sum( m * (1-dones) ), metrics))
        state = state.replace(done = dones[-1])
        return state

In [6]:
env_class, default_config = make_env(args)
env = env_class(config=default_config)
env = PDWrapper(env, duration=args.duration)
new_episode_length = default_config.episode_length / args.duration

Warp 1.9.0 initialized:
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:1"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:2"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:3"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:4"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:5"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:6"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:7"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
   CUDA peer access:
     Supported fully (all-directional)
   Kernel cache:
     /home/nvidia/.cache/warp/1.9.0


In [7]:
np.random.seed(args.seed)
key = jax.random.PRNGKey(args.seed)
key, key_env, key_eval, key_policy, key_value = jax.random.split(key, 5)

In [8]:
reset_fn = jax.jit(env.reset)
step_fn = jax.jit(env.step)

In [19]:
@jax.jit
def get_waypoint(env_state, cube_id, meta_data):
    _workspace_halfspan, _workspace_median, _yaw_halfspan, _yaw_median = meta_data
    
    current_pos = env_state.data.qpos[env._objs_qposadr[:, None] + np.arange(3)][cube_id]
    current_quat = env_state.data.qpos[(env._objs_qposadr + 3)[:, None] + np.arange(4)][cube_id]
    current_linvel = env_state.data.qvel[env._objs_qveladr[:, None] + np.arange(3)][cube_id]
    current_angvel = env_state.data.qvel[(env._objs_qveladr + 3)[:, None] + np.arange(3)][cube_id]
    
    target_pos = env_state.info['target_goal'].reshape(env._config.num_cubes, 3)[cube_id]
    
    current_xy = current_pos[:2]
    target_xy = target_pos[:2]
    
    current_height = current_pos[-1]
    top_height = target_pos[-1] + 0.05
    
    horizontal_dist_to_target = jnp.linalg.norm(current_xy - target_xy)
    dist_to_target = jnp.linalg.norm(current_pos - target_pos)

    is_far = horizontal_dist_to_target > 0.01
    is_low = current_height < (top_height - 0.005)

    wp_lift = target_pos.at[:2].set(current_xy).at[2].set(top_height)
    wp_hover = target_pos.at[2].set(top_height)
    wp_final = target_pos
        
    current_waypoint = jnp.where(
        is_far,
        jnp.where(is_low, wp_lift, wp_hover),
        wp_final
    )

    pos_action = (current_waypoint - _workspace_median) / _workspace_halfspan
    yaw_action = jnp.zeros(1)
    select_action = ( ( ( 2 * cube_id + 1) * jnp.pi / env._config.num_cubes ) - jnp.pi ) / ( jnp.pi )
    action =  jnp.concatenate([pos_action, yaw_action, select_action[None]], axis=0)
    action = jnp.clip(action, -1, 1)
    
    return action, {'dist_to_target':dist_to_target, 'target_pos': target_pos}

In [20]:
cube_mass = 0.07936
gravity = 9.81
gravity_comp = cube_mass * gravity

In [21]:
camera = mujoco.MjvCamera()
camera.distance = 0.8
camera.lookat = np.array([0.4, 0.0 , 0.4])
camera.elevation = -30.0
camera.azimuth = 180

In [25]:
returns

[Array(-80., dtype=float32),
 Array(-80., dtype=float32),
 Array(-80., dtype=float32),
 Array(-80., dtype=float32),
 Array(-80., dtype=float32),
 Array(-80., dtype=float32),
 Array(-80., dtype=float32),
 Array(-60., dtype=float32),
 Array(-60., dtype=float32),
 Array(-62., dtype=float32),
 Array(-67., dtype=float32),
 Array(-64., dtype=float32),
 Array(-64., dtype=float32),
 Array(-65., dtype=float32),
 Array(-60., dtype=float32),
 Array(-60., dtype=float32),
 Array(-60., dtype=float32),
 Array(-55., dtype=float32),
 Array(-51., dtype=float32),
 Array(-60., dtype=float32),
 Array(-40., dtype=float32),
 Array(-20., dtype=float32),
 Array(-20., dtype=float32),
 Array(-20., dtype=float32),
 Array(-9., dtype=float32)]

In [22]:
cube_id  = 0
rollout = []
returns = []
env_state = reset_fn(key_env)
rollout.append(env_state)

for i in range(int(new_episode_length)):
    action, wp_info = get_waypoint(env_state, cube_id, (env._workspace_halfspan, env._workspace_median, env._yaw_halfspan, env._yaw_median))
    if wp_info['dist_to_target'] < 0.01:
        cube_id = np.clip( cube_id + 1, a_min=0, a_max=env._config.num_cubes)

    env_state = step_fn(env_state, action)
    rollout.append(env_state)

    returns.append( env_state.reward )


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24


In [23]:
video_images = []
mocap_key = 'target_mocap'
for i in range(int(new_episode_length)):
    if i % 2 == 0:
        video_images.append(
            env.render_from_info(
                rollout[i].data.qpos,
                rollout[i].data.qvel, 
                rollout[i].info[f'{mocap_key}_pos'],
                rollout[i].info[f'{mocap_key}_quat'],
                camera=camera,
            )
        )

In [24]:
media.show_video(video_images, fps=1.0 / env.dt / 2)

In [19]:
default_config.episode_length

500